# Statistical Analysis with Pandas

## Introduction

Pandas provides built-in methods to compute descriptive statistics, correlations, and group-level summaries. These tools bridge the gap between raw data and insights — and form the backbone of exploratory data analysis (EDA).

## Objectives

You will be able to:

* Use `.describe()`, `.value_counts()`, and aggregation methods
* Group data with `.groupby()` and compute group statistics
* Compute correlations between columns
* Apply `.apply()` and `.transform()` for custom computations
* Sort and rank data

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

np.random.seed(42)

# Employee dataset
n = 100
depts = np.random.choice(['Engineering', 'Marketing', 'Data Science', 'Sales'], n)
df = pd.DataFrame({
    'dept':       depts,
    'age':        np.random.randint(22, 55, n),
    'salary':     np.random.normal(75000, 15000, n).round(-2).astype(int),
    'experience': np.random.randint(1, 20, n),
    'rating':     np.random.uniform(2.5, 5.0, n).round(1),
})
df['salary'] = df['salary'].clip(40000, 150000)
print(df.head(5))
print(f"Shape: {df.shape}")

---

## Descriptive Statistics

In [ ]:
# .describe() — summary for all numeric columns
print(df.describe().round(1))

In [ ]:
# Include categorical columns
print(df.describe(include='all'))

In [ ]:
# Individual statistics
print(f"Mean salary:    ${df['salary'].mean():,.0f}")
print(f"Median salary:  ${df['salary'].median():,.0f}")
print(f"Std salary:     ${df['salary'].std():,.0f}")
print(f"Min/Max:        ${df['salary'].min():,} / ${df['salary'].max():,}")

# Quantiles
print(f"\nSalary percentiles:")
print(df['salary'].quantile([0.25, 0.5, 0.75]))

In [ ]:
# value_counts — frequency of categorical values
print(df['dept'].value_counts())
print()
print(df['dept'].value_counts(normalize=True).round(2))  # as proportions

---

## GroupBy — Group-Level Statistics

In [ ]:
# Mean salary by department
print(df.groupby('dept')['salary'].mean().sort_values(ascending=False))

In [ ]:
# Multiple statistics with .agg()
dept_stats = df.groupby('dept')['salary'].agg(['mean', 'median', 'std', 'count'])
dept_stats.columns = ['mean', 'median', 'std', 'count']
dept_stats = dept_stats.round(0).astype(int)
print(dept_stats)

In [ ]:
# Named aggregations (pandas >= 0.25)
summary = df.groupby('dept').agg(
    avg_salary=('salary', 'mean'),
    headcount=('salary', 'count'),
    avg_rating=('rating', 'mean'),
    avg_experience=('experience', 'mean'),
).round(1)
print(summary)

In [ ]:
# GroupBy multiple columns
senior_flag = pd.cut(df['experience'], bins=[0, 5, 10, 20], labels=['junior', 'mid', 'senior'])
df['seniority'] = senior_flag

pivot = df.groupby(['dept', 'seniority'])['salary'].mean().round(0).unstack()
print(pivot)

---

## Correlation

In [ ]:
# Correlation matrix for all numeric columns
corr = df[['age', 'salary', 'experience', 'rating']].corr()
print(corr.round(2))

In [ ]:
# Visualize as a heatmap
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
numeric_cols = ['age', 'salary', 'experience', 'rating']
corr_mat = df[numeric_cols].corr().values
im = ax.imshow(corr_mat, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right')
ax.set_yticklabels(numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f'{corr_mat[i, j]:.2f}', ha='center', va='center', fontsize=10)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

---

## Sorting and Ranking

In [ ]:
# Sort by one column
print(df.sort_values('salary', ascending=False).head(5)[['dept', 'salary', 'experience']])

# Sort by multiple columns
print(df.sort_values(['dept', 'salary'], ascending=[True, False]).head(8)[['dept', 'salary']])

# Rank within the full dataset
df['salary_rank'] = df['salary'].rank(method='dense', ascending=False).astype(int)
print(df.sort_values('salary_rank').head(5)[['dept', 'salary', 'salary_rank']])

---

## `.apply()` — Custom Functions on Columns or Rows

In [ ]:
# Apply a function to each value in a column
def salary_band(salary):
    if salary < 60000:   return 'Low'
    elif salary < 90000: return 'Mid'
    else:                return 'High'

df['salary_band'] = df['salary'].apply(salary_band)
print(df['salary_band'].value_counts())

# Lambda version — for simple transformations
df['rating_5'] = df['rating'].apply(lambda x: round(x * 2) / 2)  # round to 0.5

In [ ]:
# .transform() — like apply but returns a Series with the same index
# Useful for adding group-level stats back to the original DataFrame
df['dept_avg_salary'] = df.groupby('dept')['salary'].transform('mean').round(0).astype(int)
df['above_dept_avg'] = df['salary'] > df['dept_avg_salary']

print(df[['dept', 'salary', 'dept_avg_salary', 'above_dept_avg']].head(10))
print(f"\n{df['above_dept_avg'].sum()} employees above their department average")

---

## Practice

In [ ]:
# 1. Which department has the highest average rating?
top_dept = None
print(top_dept)

In [ ]:
# 2. What is the correlation between experience and salary?
corr_val = None
print(f"Experience-Salary correlation: {corr_val:.3f}")

In [ ]:
# 3. Add a 'pay_percentile' column showing each employee's salary percentile rank (0-100)
# Hint: use .rank(pct=True)
df['pay_percentile'] = None
print(df[['dept', 'salary', 'pay_percentile']].sort_values('pay_percentile', ascending=False).head(5))

## Summary

| Task | Method |
|------|--------|
| Summary stats | `.describe()`, `.mean()`, `.std()`, `.quantile()` |
| Categorical counts | `.value_counts()`, `.value_counts(normalize=True)` |
| Group statistics | `.groupby('col').agg(...)` |
| Correlation matrix | `.corr()` |
| Custom per-row | `.apply(func)` |
| Group-level back-fill | `.groupby().transform()` |

Next: data ethics — understanding bias and fairness before modeling.